In [55]:
# ==========================================
# 0. ИМПОРТЫ
# ==========================================
import re
import time
import pickle
import requests
import numpy as np
import faiss
import fitz  # PyMuPDF
import os
from pathlib import Path
import subprocess

from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
# OPENAI
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
# ==========================================
# 1. КОНФИГ
# ==========================================
INDEX_PATH = "my_index.faiss"
CHUNKS_PATH = "my_chunks.pkl"

OLLAMA_BASE_URL = "http://localhost:11434"
EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "gpt-oss:120b-cloud"

VERBOSE = True

# OPENAI

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("Переменная OPENAI_API_KEY не найдена")

client = OpenAI(api_key=OPENAI_API_KEY)

In [3]:
# ==========================================
# 2. ЛОГИРОВАНИЕ
# ==========================================
def log(*args):
    if VERBOSE:
        print(*args)

In [4]:
# ==========================================
# 3. OLLAMA CHAT
# ==========================================
def ollama_chat(system: str, user: str, model: str = LLM_MODEL) -> str:

    prompt = f"""
{system}

{user}
"""

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.2,
            "top_p": 0.9
        }
    }

    r = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json=payload,
        timeout=300
    )

    r.raise_for_status()

    data = r.json()

    if "response" not in data:
        raise ValueError(f"Некорректный ответ Ollama: {data}")

    return data["response"].strip()

In [5]:
# ==========================================
# 4. ЭМБЕДДИНГ ЧЕРЕЗ OLLAMA
# ==========================================
def get_embedding(text: str) -> np.ndarray:
    text = (text or "").strip()

    if not text:
        raise ValueError("Пустой текст для embedding")

    payload = {
        "model": EMBED_MODEL,
        "prompt": text
    }

    r = requests.post(
        f"{OLLAMA_BASE_URL}/api/embeddings",
        json=payload,
        timeout=300
    )
    r.raise_for_status()

    data = r.json()

    if "embedding" not in data:
        raise ValueError(f"Нет embedding в ответе Ollama: {data}")

    return np.array(data["embedding"], dtype="float32")

In [6]:
# ==========================================
# 5. ПРОВЕРКА: ЭТО ПЕРВАЯ ГЛАВА?
# ==========================================
def is_first_chapter_page(text: str) -> bool:
    if not text:
        return False

    text = text.strip().lower()

    patterns = [
        r"\bchapter\s*1\b",
        r"\bглава\s*1\b",
        r"^\s*1[\.\s]+\w+",
        r"\bchapter one\b",
    ]

    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)

In [7]:
# ==========================================
# 6. НАЙТИ СТРАНИЦУ ПЕРВОЙ ГЛАВЫ
# ==========================================
def find_first_chapter_page(doc, max_pages_to_scan=None):
    if max_pages_to_scan is None:
        max_pages_to_scan = len(doc)

    limit = min(max_pages_to_scan, len(doc))
    log(f"Ищу первую главу в первых {limit} страницах...")

    for i in range(limit):
        text = doc[i].get_text("text")
        if is_first_chapter_page(text):
            log(f"Первая глава найдена на странице: {i + 1}")
            return i

    log("Первая глава не найдена. Беру с начала документа.")
    return 0

In [8]:
# ==========================================
# 7. ПРОВЕРКА: СТРАНИЦА ОГЛАВЛЕНИЯ
# ==========================================
def is_contents_page(text: str) -> bool:
    if not text:
        return False

    low = text.lower()
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    if not lines:
        return False

    contents_keywords = [
        "contents",
        "table of contents",
        "оглавление",
        "содержание"
    ]
    if any(k in low for k in contents_keywords):
        return True

    section_only_count = 0
    page_only_count = 0
    dotted_count = 0
    short_title_count = 0
    section_title_count = 0
    long_paragraph_count = 0
    formula_number_count = 0
    math_symbol_count = 0

    for line in lines[:200]:
        if len(line) >= 90 and len(line.split()) >= 8:
            long_paragraph_count += 1

        if re.fullmatch(r"\(\d+(\.\d+)*\)", line):
            formula_number_count += 1

        if re.search(r"[=∑πωΩ±√∫]", line) or "exp(" in line.lower():
            math_symbol_count += 1

        if re.fullmatch(r"\d+(\.\d+)*", line):
            if "." in line:
                section_only_count += 1
            else:
                if int(line) <= 20:
                    section_only_count += 1
                else:
                    page_only_count += 1
            continue

        if re.fullmatch(r"\d{1,4}", line):
            page_only_count += 1
            continue

        if re.fullmatch(r"\.+", line) or re.fullmatch(r"(\.\s*){4,}\d*", line):
            dotted_count += 1
            continue

        if re.match(r"^\d+(\.\d+)*\s+\S+", line):
            section_title_count += 1
            continue

        words = line.split()
        if 1 <= len(words) <= 12 and len(line) <= 120:
            short_title_count += 1

    if long_paragraph_count >= 2:
        return False

    if formula_number_count >= 2:
        return False

    if math_symbol_count >= 2:
        return False

    if section_only_count >= 5 and page_only_count >= 5:
        return True

    if section_only_count >= 4 and dotted_count >= 4 and short_title_count >= 4:
        return True

    if page_only_count >= 5 and dotted_count >= 5:
        return True

    if section_title_count >= 5 and page_only_count >= 4:
        return True

    return False

In [9]:
# ==========================================
# 8. СКЛЕЙКА СТРОК ИЗ BLOCKS
# ==========================================
def merge_toc_lines_from_blocks(blocks):
    items = []

    for b in blocks:
        if b.get("type", 0) != 0:
            continue

        if "lines" not in b:
            continue

        for line in b["lines"]:
            spans = line.get("spans", [])
            if not spans:
                continue

            line_text = " ".join(
                (span.get("text") or "").strip()
                for span in spans
                if (span.get("text") or "").strip()
            ).strip()

            if not line_text:
                continue

            bbox = line["bbox"] if "bbox" in line else b["bbox"]
            y0 = bbox[1]
            x0 = bbox[0]

            items.append((y0, x0, line_text))

    items.sort(key=lambda x: (round(x[0], 1), x[1]))
    return [text for _, _, text in items]

In [10]:
# ==========================================
# 9. ПРОВЕРКА: СТРАНИЦА INDEX
# ==========================================
def is_index_page(blocks) -> bool:
    lines = merge_toc_lines_from_blocks(blocks)
    pattern = r"^[A-Za-zА-Яа-яЁё][A-Za-zА-Яа-яЁё0-9\-() /]+,\s*\d+(,\s*\d+)*$"
    index_score = sum(1 for text in lines if re.match(pattern, text.strip()))
    return index_score >= 5

In [11]:
# ==========================================
# 10. ПРОВЕРКА: ПОХОЖ ЛИ ТЕКСТ НА ФОРМУЛУ
# ==========================================
def looks_like_math(text: str) -> bool:
    if not text:
        return False

    text = text.strip()

    if len(text) > 120:
        return False

    math_tokens = [
        "=", "∫", "π", "δ", "√", "∞",
        "sin", "cos", "tan", "exp",
        "dx", "dy", "dz",
        "∑", "±", "≤", "≥", "ω", "λ", "μ"
    ]

    if any(tok in text for tok in math_tokens):
        return True

    if re.search(r"\(\d+(\.\d+)*\)$", text):
        return True

    special_count = sum(ch in "=+-*/^_[]{}<>∫πδ√∞∑±≤≥ωλμ" for ch in text)
    if special_count >= 3:
        return True

    return False

In [12]:
# ==========================================
# 11. ЖИРНЫЙ ЛИ ШРИФТ
# ==========================================
def is_bold_font(font_name: str, flags: int = 0) -> bool:
    font_name = (font_name or "").lower()

    if "bold" in font_name:
        return True

    if isinstance(flags, int) and (flags & 16):
        return True

    return False

In [13]:
# ==========================================
# 12. КЛАССИФИКАЦИЯ БЛОКА
# ==========================================
def classify_block(text: str, max_size: float, avg_size: float, bold_ratio: float) -> str:
    text = (text or "").strip()

    if not text:
        return "empty"

    if looks_like_math(text):
        return "formula"

    text_len = len(text)

    if max_size >= 16 and text_len <= 150:
        return "heading_1"

    if max_size >= 14 and text_len <= 180:
        return "heading_2"

    if max_size >= 12.5 and bold_ratio >= 0.5 and text_len <= 220:
        return "heading_3"

    return "text"

In [14]:
# ==========================================
# 13. ИЗВЛЕЧЕНИЕ И ОЧИСТКА БЛОКОВ СО СТРАНИЦЫ
# ==========================================
def extract_text_blocks(page, drop_headers=True, header_ratio=0.08, footer_ratio=0.06):
    page_height = page.rect.height
    raw_blocks = page.get_text("dict").get("blocks", [])

    blocks = []

    for b in raw_blocks:
        if b.get("type", 0) != 0:
            continue

        if "lines" not in b:
            continue

        x0, y0, x1, y1 = b["bbox"]

        if drop_headers:
            if y0 < page_height * header_ratio:
                continue
            if y1 > page_height * (1 - footer_ratio):
                continue

        text_parts = []
        spans_info = []

        for line in b["lines"]:
            for span in line.get("spans", []):
                span_text = (span.get("text") or "").strip()
                if not span_text:
                    continue

                size = span.get("size", 0)
                font = span.get("font", "")
                flags = span.get("flags", 0)
                bbox = span.get("bbox")

                text_parts.append(span_text)

                spans_info.append({
                    "text": span_text,
                    "size": size,
                    "font": font,
                    "flags": flags,
                    "bbox": bbox,
                    "is_bold": is_bold_font(font, flags),
                })

        text = " ".join(text_parts).strip()

        if not text:
            continue

        sizes = [s["size"] for s in spans_info if s["size"] is not None]
        max_size = max(sizes) if sizes else 0
        avg_size = sum(sizes) / len(sizes) if sizes else 0

        bold_count = sum(1 for s in spans_info if s["is_bold"])
        bold_ratio = bold_count / len(spans_info) if spans_info else 0

        block_type = classify_block(
            text=text,
            max_size=max_size,
            avg_size=avg_size,
            bold_ratio=bold_ratio
        )

        blocks.append({
            "text": text,
            "bbox": b["bbox"],
            "page_num": page.number,
            "type": block_type,
            "max_size": max_size,
            "avg_size": avg_size,
            "bold_ratio": bold_ratio,
            "spans": spans_info,
        })

    return blocks

In [15]:
# ==========================================
# 14. ПОЛУЧИТЬ СПИСОК НУЖНЫХ СТРАНИЦ
# ==========================================
def get_working_page_numbers(doc, max_pages_to_scan=None):
    start_page = find_first_chapter_page(doc, max_pages_to_scan=max_pages_to_scan)

    keep_pages = []
    skipped_contents = []
    skipped_index = []

    log("Начинаю фильтрацию страниц...")

    for i in range(start_page, len(doc)):
        page = doc[i]

        text = page.get_text("text")
        blocks = page.get_text("dict")["blocks"]

        if is_contents_page(text):
            skipped_contents.append(i)
            log(f"Пропуск CONTENTS: стр. {i + 1}")
            continue

        if is_index_page(blocks):
            skipped_index.append(i)
            log(f"Пропуск INDEX:    стр. {i + 1}")
            continue

        keep_pages.append(i)

    log(f"Страниц оставлено: {len(keep_pages)}")
    log(f"Пропущено contents: {len(skipped_contents)}")
    log(f"Пропущено index:    {len(skipped_index)}")

    return keep_pages

In [16]:
# ==========================================
# 15. СОБРАТЬ БЛОКИ ТОЛЬКО С НУЖНЫХ СТРАНИЦ
# ==========================================
def extract_blocks_from_working_pages(doc, working_pages):
    all_blocks = []

    log("Начинаю извлечение блоков...")

    for n, page_num in enumerate(working_pages, start=1):
        page = doc[page_num]
        page_blocks = extract_text_blocks(page)

        for b in page_blocks:
            b["page_num"] = page_num

        all_blocks.extend(page_blocks)

        log(f"[{n}/{len(working_pages)}] стр. {page_num + 1} -> блоков: {len(page_blocks)}")

    log(f"Всего извлечено блоков: {len(all_blocks)}")
    return all_blocks

In [17]:
# ==========================================
# 16. СТРУКТУРНЫЕ ЧАНКИ С МЕТАДАННЫМИ
# ==========================================
def chunk_blocks_with_metadata(blocks, max_chars=1200, overlap=200, source_name="unknown_source"):
    chunks = []

    current_chapter = None
    current_section = None
    current_subsection = None

    current_text_parts = []
    current_pages = []
    current_formulas = []
    current_block_types = []

    def flush_chunk():
        nonlocal current_text_parts, current_pages, current_formulas, current_block_types

        text = "\n".join(part for part in current_text_parts if part.strip()).strip()
        if not text:
            return

        chunks.append({
            "text": text,
            "pages": sorted(set(current_pages)),
            "metadata": {
                "source": source_name,
                "chapter": current_chapter,
                "section": current_section,
                "subsection": current_subsection,
                "block_types": sorted(set(current_block_types)),
                "formulas": current_formulas.copy(),
            }
        })

        current_text_parts = []
        current_pages = []
        current_formulas = []
        current_block_types = []

    log("Начинаю сборку структурных чанков...")

    for i, b in enumerate(blocks, start=1):
        text = (b.get("text") or "").strip()
        block_type = b.get("type", "text")
        page_num = b.get("page_num")

        if not text:
            continue

        if block_type == "heading_1":
            flush_chunk()
            current_chapter = text
            current_section = None
            current_subsection = None
            log(f"heading_1 -> {text}")
            continue

        if block_type == "heading_2":
            flush_chunk()
            current_section = text
            current_subsection = None
            log(f"heading_2 -> {text}")
            continue

        if block_type == "heading_3":
            flush_chunk()
            current_subsection = text
            log(f"heading_3 -> {text}")
            continue

        if block_type == "formula":
            current_formulas.append(text)
            current_block_types.append("formula")

            formula_text = f"[FORMULA]\n{text}\n[/FORMULA]"
            candidate_text = "\n".join(current_text_parts + [formula_text]).strip()

            if len(candidate_text) > max_chars and current_text_parts:
                flush_chunk()

            current_text_parts.append(formula_text)
            current_pages.append(page_num)
            continue

        current_block_types.append(block_type)

        candidate_text = "\n".join(current_text_parts + [text]).strip()

        if len(candidate_text) <= max_chars:
            current_text_parts.append(text)
            current_pages.append(page_num)
        else:
            flush_chunk()

            if overlap > 0 and chunks:
                prev_text = chunks[-1]["text"]
                overlap_text = prev_text[-overlap:] if len(prev_text) > overlap else prev_text
                if overlap_text.strip():
                    current_text_parts.append(overlap_text)

            current_text_parts.append(text)
            current_pages.append(page_num)

        if i % 100 == 0:
            log(f"Обработано блоков: {i}/{len(blocks)} | чанков сейчас: {len(chunks)}")

    flush_chunk()
    log(f"Всего создано чанков: {len(chunks)}")

    return chunks

In [18]:
# ==========================================
# 17. ПОЛНЫЙ ПАЙПЛАЙН: PDF -> ЧАНКИ
# ==========================================
def build_chunks_from_pdf(doc, source_name="book.pdf", max_pages_to_scan=None, max_chars=1200, overlap=200):
    working_pages = get_working_page_numbers(doc, max_pages_to_scan=max_pages_to_scan)
    log("Страницы после очистки:", len(working_pages))

    all_blocks = extract_blocks_from_working_pages(doc, working_pages)
    log("Всего блоков:", len(all_blocks))

    chunks = chunk_blocks_with_metadata(
        all_blocks,
        max_chars=max_chars,
        overlap=overlap,
        source_name=source_name
    )
    log("Всего чанков:", len(chunks))

    return chunks

In [19]:
# ==========================================
# 18. ЧАНКИ -> ЭМБЕДДИНГИ
# ==========================================
def build_embeddings_for_chunks(chunks):
    vectors = []
    enriched_chunks = []

    log("Начинаю создание эмбеддингов...")

    total = len(chunks)

    for i, chunk in enumerate(chunks):
        text = chunk["text"]
        emb = get_embedding(text)

        vectors.append(emb)

        enriched_chunk = {
            "chunk_id": i,
            "text": text,
            "pages": chunk.get("pages", []),
            "metadata": chunk.get("metadata", {})
        }

        enriched_chunks.append(enriched_chunk)

        if (i + 1) % 10 == 0 or (i + 1) == total:
            log(f"Эмбеддинги: {i + 1}/{total}")

    vectors = np.array(vectors, dtype="float32")
    log("Создание эмбеддингов завершено.")
    log("Форма массива vectors:", vectors.shape)

    return enriched_chunks, vectors

In [20]:
# ==========================================
# 19. СОЗДАНИЕ FAISS ИНДЕКСА
# ==========================================
def build_faiss_index(vectors: np.ndarray):
    if len(vectors.shape) != 2:
        raise ValueError(f"Ожидался 2D массив vectors, получено: {vectors.shape}")

    log("Начинаю построение FAISS индекса...")

    dim = vectors.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(vectors)

    log(f"FAISS индекс готов. Векторов в базе: {index.ntotal}")
    return index

In [21]:
# ==========================================
# 20. СОХРАНЕНИЕ БАЗЫ
# ==========================================
def save_knowledge_base(index, chunks, index_path=INDEX_PATH, chunks_path=CHUNKS_PATH):
    log("Сохраняю FAISS индекс...")
    faiss.write_index(index, index_path)

    log("Сохраняю чанки...")
    with open(chunks_path, "wb") as f:
        pickle.dump(chunks, f)

    log("База сохранена:")
    log("FAISS:", index_path)
    log("CHUNKS:", chunks_path)

In [22]:
# ==========================================
# 21. ЗАГРУЗКА БАЗЫ
# ==========================================
def load_knowledge_base(index_path=INDEX_PATH, chunks_path=CHUNKS_PATH):
    log("Загружаю FAISS индекс...")
    index = faiss.read_index(index_path)

    log("Загружаю чанки...")
    with open(chunks_path, "rb") as f:
        chunks = pickle.load(f)

    log("База загружена:")
    log("FAISS:", index_path)
    log("CHUNKS:", chunks_path)

    return index, chunks

In [23]:
# ==========================================
# 22. ПОИСК ПО БАЗЕ
# ==========================================
def search_in_db(query: str, index=None, all_chunks=None, k: int = 5):
    global GLOBAL_INDEX, GLOBAL_CHUNKS

    if index is None:
        index = GLOBAL_INDEX

    if all_chunks is None:
        all_chunks = GLOBAL_CHUNKS

    if index is None or all_chunks is None:
        print("База не загружена.")
        return []

    log(f"Поиск по базе. Запрос: {query}")

    q = get_embedding(query).astype("float32").reshape(1, -1)

    if q.shape[1] != index.d:
        print("Ошибка: размерность embedding запроса не совпадает с базой.")
        print("Размерность запроса:", q.shape[1])
        print("Размерность базы:", index.d)
        return []

    D, I = index.search(q, k)

    results = []

    for dist, idx in zip(D[0], I[0]):
        if idx == -1:
            continue

        chunk_data = all_chunks[idx]
        metadata = chunk_data.get("metadata", {})

        results.append({
            "idx": int(idx),
            "distance": float(dist),
            "text": chunk_data.get("text", ""),
            "pages": chunk_data.get("pages", []),
            "source": metadata.get("source"),
            "chapter": metadata.get("chapter"),
            "section": metadata.get("section"),
            "subsection": metadata.get("subsection"),
            "formulas": metadata.get("formulas", []),
            "block_types": metadata.get("block_types", []),
        })

    log(f"Найдено результатов: {len(results)}")
    return results

In [24]:
# ==========================================
# 23. LANGGRAPH: СОСТОЯНИЕ ДЕБАТОВ
# ==========================================
class DebateState(TypedDict):
    question: str
    retrieved_context: str
    retrieved_items: List[dict]
    hypothesis: str
    criticism: str
    evidence: str
    final_answer: str

In [25]:
# ==========================================
# 24. ПОДГОТОВКА КОНТЕКСТА ДЛЯ ДЕБАТОВ
# ==========================================
def build_context_from_results(results: list[dict]) -> str:
    if not results:
        return ""

    parts = []

    for i, item in enumerate(results, start=1):
        source = item.get("source", "unknown_source")
        chapter = item.get("chapter") or "—"
        section = item.get("section") or "—"
        subsection = item.get("subsection") or "—"
        pages = item.get("pages", [])
        distance = item.get("distance", 0.0)
        text = item.get("text", "").strip()
        formulas = item.get("formulas", [])

        page_str = ", ".join(map(str, pages)) if pages else "—"

        formulas_block = "нет"
        if formulas:
            formulas_block = "\n".join(f"- {f}" for f in formulas[:5])

        part = (
            f"[ФРАГМЕНТ {i}]\n"
            f"Источник: {source}\n"
            f"Глава: {chapter}\n"
            f"Раздел: {section}\n"
            f"Подраздел: {subsection}\n"
            f"Страницы: {page_str}\n"
            f"Distance: {distance:.4f}\n"
            f"Формулы:\n{formulas_block}\n"
            f"Текст:\n{text}\n"
        )
        parts.append(part)

    return "\n\n".join(parts)

In [26]:
# ==========================================
# 25. УЗЕЛ: ИЗВЛЕЧЕНИЕ КОНТЕКСТА ИЗ БАЗЫ
# ==========================================
def retrieve_context_node(state: DebateState) -> dict:
    question = state["question"]
    log("Debate node: retrieve_context")

    results = search_in_db(question, k=5)

    if not results:
        return {
            "retrieved_items": [],
            "retrieved_context": ""
        }

    context = build_context_from_results(results)

    return {
        "retrieved_items": results,
        "retrieved_context": context
    }

In [27]:
# ==========================================
# 26. УЗЕЛ: ВЫДВИЖЕНИЕ ГИПОТЕЗЫ
# ==========================================
def build_hypothesis_node(state: DebateState) -> dict:
    log("Debate node: build_hypothesis")

    question = state["question"]
    context = state.get("retrieved_context", "")

    if not context.strip():
        return {
            "hypothesis": "Гипотеза не может быть сформулирована, потому что в базе не найден релевантный контекст."
        }

    system = """
Ты аналитик RAG-системы.

Твоя задача:
- прочитать вопрос;
- прочитать контекст;
- сформулировать предварительную гипотезу;
- опираться только на контекст;
- не выдумывать факты;
- не использовать знания вне контекста;
- если данных мало, прямо скажи об этом.

Пиши на русском языке.
Выводи только текст гипотезы.
"""

    user = f"""
Вопрос:
{question}

Контекст:
{context}

Сформулируй предварительную гипотезу по вопросу строго на основе контекста.
"""

    hypothesis = ollama_chat(system=system, user=user)
    return {"hypothesis": hypothesis}

In [28]:
# ==========================================
# 27. УЗЕЛ: КРИТИКА ГИПОТЕЗЫ
# ==========================================
def critic_review_node(state: DebateState) -> dict:
    log("Debate node: critic_review")

    question = state["question"]
    context = state.get("retrieved_context", "")
    hypothesis = state.get("hypothesis", "")

    if not context.strip():
        return {
            "criticism": "Критика невозможна, потому что отсутствует найденный контекст."
        }

    system = """
Ты критик-аналитик.

Твоя задача:
- проверить гипотезу на слабые места;
- указать, что подтверждено хорошо;
- указать, что подтверждено слабо;
- отметить противоречия;
- отметить, каких данных не хватает;
- не выдумывать факты;
- использовать только переданный контекст.

Пиши на русском языке.
Выводи только критический разбор.
"""

    user = f"""
Вопрос:
{question}

Контекст:
{context}

Гипотеза:
{hypothesis}

Проведи критический анализ гипотезы.
"""

    criticism = ollama_chat(system=system, user=user)
    return {"criticism": criticism}

In [36]:
# ==========================================
# 28. УЗЕЛ: ДОКАЗАТЕЛЬСТВА
# ==========================================
def evidence_check_node(state: DebateState) -> dict:
    log("Debate node: evidence_check")

    question = state["question"]
    context = state.get("retrieved_context", "")
    hypothesis = state.get("hypothesis", "")
    criticism = state.get("criticism", "")

    if not context.strip():
        return {
            "evidence": "Доказательства отсутствуют, потому что не найден контекст в базе."
        }

    system = """
Ты аналитик доказательств.

Твоя задача:
- выделить подтверждающие доказательства;
- выделить ослабляющие или опровергающие доказательства;
- ссылаться только на переданные фрагменты;
- не придумывать факты;
- если данных недостаточно, прямо укажи это.

Пиши на русском языке.

Формат:
Подтверждает:
- ...

Ослабляет или опровергает:
- ...
"""

    user = f"""
Вопрос:
{question}

Контекст:
{context}

Гипотеза:
{hypothesis}

Критика:
{criticism}

Выдели подтверждающие и ослабляющие доказательства.
"""

    evidence = ollama_chat(system=system, user=user)
    return {"evidence": evidence}

In [37]:
# ==========================================
# 29. УЗЕЛ: ФИНАЛЬНЫЙ ВЫВОД
# ==========================================
def final_conclusion_node(state: DebateState) -> dict:
    question = state["question"]
    context = state.get("retrieved_context", "")
    hypothesis = state.get("hypothesis", "")
    criticism = state.get("criticism", "")
    evidence = state.get("evidence", "")

    system = f"""
Ты научный ассистент и финальный аналитик RAG-системы.

Твоя задача:
- ответить на вопрос пользователя;
- учесть контекст из базы знаний;
- учесть гипотезу;
- учесть критику;
- учесть подтверждающие или опровергающие факты;
- сформировать понятный, логичный и аккуратно оформленный итоговый ответ.

# ГЛАВНОЕ ТРЕБОВАНИЕ

Ответ должен корректно отображаться в Markdown (Jupyter Notebook),
включая ВСЕ математические формулы.

# КРИТИЧЕСКИ ВАЖНЫЕ ПРАВИЛА (НЕ НАРУШАТЬ)

1. ЛЮБОЕ математическое выражение ВСЕГДА должно быть обёрнуто:
   - либо в $ ... $ (внутри текста)
   - либо в $$ ... $$ (отдельная формула)

❌ НЕЛЬЗЯ:
(\text{eff} \approx 1)
mu(mathbf(x))
lambda

✅ ТОЛЬКО ТАК:
($n_{{eff}} \approx 1$)
$\mu(\mathbf{{x}})$
$\lambda$

---

2. ЗАПРЕЩЕНО использовать LaTeX без $...$

❌ НЕЛЬЗЯ:
текст ... \lambda ...
текст ... \mu ...

✅ ВСЕГДА:
текст ... $\lambda$ ...
текст ... $\mu$ ...

---

3. Формулы внутри текста:

ВСЕГДА:
... оболочка с показателем преломления $n_{{eff}} \approx 1$ ...

Если формула в скобках:
... ( $n_{{eff}} \approx 1$ ) ...

---

4. Отдельные формулы:

$$
\\nabla \\cdot \\mathbf{{E}} = \\frac{{\\rho}}{{\\varepsilon_0}}
$$

---

5. ЗАПРЕЩЕНО:
- Unicode символы (π, μ, √, Δ)
- псевдо-LaTeX (mathbf(), text())
- кодовые блоки ``` ```
- обратные кавычки `...`

---

6. Строго использовать LaTeX:

- дроби: \\frac{{a}}{{b}}
- индексы: x_i
- степени: x^2
- векторы: \\mathbf{{E}}
- функции: \\sin, \\cos
- интегралы: \\int
- длины волн: \\lambda
- диэлектрическая проницаемость: \\varepsilon
- магнитная проницаемость: \\mu

---

7. ОСОБО ВАЖНО (исправляет твою ошибку):

Если формула находится внутри предложения —
она НЕ должна "ломать" текст.

❌ ПЛОХО:
... коэффициент (\\text{{eff}} \\approx 1) ...

❌ ПЛОХО:
mu(mathbf(x))

✅ ПРАВИЛЬНО:
... коэффициент $n_{{eff}} \\approx 1$ ...
... функция $\\mu(\\mathbf{{x}})$ ...

---

# СТРУКТУРА ОТВЕТА

ГИПОТЕЗА  
...

КРИТИКА ГИПОТЕЗЫ  
...

ДОКАЗАТЕЛЬСТВА  
Подтверждает:
- ...
Ослабляет или опровергает:
- ...

ФОРМУЛЫ ИЗ ИСТОЧНИКОВ  
- ...

ИСТОЧНИКИ  
- [Фрагмент N] Источник: ... | Глава: ... | Раздел: ... | Страницы: ...

ИТОГОВЫЙ ВЫВОД  
...

УРОВЕНЬ УВЕРЕННОСТИ  
высокий / средний / низкий

# ВАЖНО

- Никаких комментариев о форматировании
- Просто выдай чистый результат
- Все формулы — только в LaTeX и только в $ или $$



Формат ответа строго такой:

ГИПОТЕЗА
...

КРИТИКА ГИПОТЕЗЫ
...

ДОКАЗАТЕЛЬСТВА
Подтверждает:
- ...
Ослабляет или опровергает:
- ...

ФОРМУЛЫ ИЗ ИСТОЧНИКОВ
- ...

ИСТОЧНИКИ
- [Фрагмент N] Источник: ... | Глава: ... | Раздел: ... | Страницы: ...

ИТОГОВЫЙ ВЫВОД
...

УРОВЕНЬ УВЕРЕННОСТИ
высокий / средний / низкий
"""

    user = f"""
Вопрос:
{question}

Контекст:
{context}

Гипотеза:
{hypothesis}

Критика:
{criticism}

Доказательства:
{evidence}

Сформируй финальный ответ строго по формату.
"""

    final_answer = ollama_chat(system=system, user=user)

    return {"final_answer": final_answer}

In [38]:
# ==========================================
# 30. СБОРКА LANGGRAPH-ГРАФА
# ==========================================
def build_debate_graph():
    log("Собираю debate graph...")

    graph = StateGraph(DebateState)

    graph.add_node("retrieve_context", retrieve_context_node)
    graph.add_node("build_hypothesis", build_hypothesis_node)
    graph.add_node("critic_review", critic_review_node)
    graph.add_node("evidence_check", evidence_check_node)
    graph.add_node("final_conclusion", final_conclusion_node)

    graph.add_edge(START, "retrieve_context")
    graph.add_edge("retrieve_context", "build_hypothesis")
    graph.add_edge("build_hypothesis", "critic_review")
    graph.add_edge("critic_review", "evidence_check")
    graph.add_edge("evidence_check", "final_conclusion")
    graph.add_edge("final_conclusion", END)

    app = graph.compile()
    log("Debate graph готов.")
    return app

In [39]:
# ==========================================
# 31. ЗАПУСК ДЕБАТНОГО РЕЖИМА
# ==========================================
def ask_debate_rag(question: str) -> str:
    global debate_app

    if debate_app is None:
        raise ValueError("Debate graph не собран. Сначала построй или загрузи базу.")

    log("Запуск debate RAG...")

    state = {
        "question": question,
        "retrieved_context": "",
        "retrieved_items": [],
        "hypothesis": "",
        "criticism": "",
        "evidence": "",
        "final_answer": ""
    }

    result = debate_app.invoke(state)
    return result["final_answer"]

In [40]:
# ==========================================
# 32. ГЛОБАЛЬНЫЕ ПЕРЕМЕННЫЕ БАЗЫ
# ==========================================
GLOBAL_INDEX = None
GLOBAL_CHUNKS = []
debate_app = None

if os.path.exists(INDEX_PATH) and os.path.exists(CHUNKS_PATH):
    print("Найдена сохранённая база. Загружаю...")

    GLOBAL_INDEX, GLOBAL_CHUNKS = load_knowledge_base(INDEX_PATH, CHUNKS_PATH)
    debate_app = build_debate_graph()

    print("База загружена.")
    print("Чанков:", len(GLOBAL_CHUNKS))
    print("Размер индекса:", GLOBAL_INDEX.ntotal)
else:
    print("Файлы базы не найдены.")

Найдена сохранённая база. Загружаю...
Загружаю FAISS индекс...
Загружаю чанки...
База загружена:
FAISS: my_index.faiss
CHUNKS: my_chunks.pkl
Собираю debate graph...
Debate graph готов.
База загружена.
Чанков: 3995
Размер индекса: 3995


In [41]:
# ==========================================
# 33. ПОКАЗАТЬ КНИГИ, КОТОРЫЕ УЖЕ ДОБАВЛЕНЫ В БАЗУ
# ==========================================
def list_books():
    global GLOBAL_CHUNKS

    if not GLOBAL_CHUNKS:
        print("База пустая")
        return

    books = set()

    for chunk in GLOBAL_CHUNKS:
        if not isinstance(chunk, dict):
            continue

        metadata = chunk.get("metadata", {})
        source = metadata.get("source")
        if source:
            books.add(source)
            continue

        if "book" in chunk and chunk["book"]:
            books.add(chunk["book"])

    books = sorted(books)

    print("\nКниги в базе:\n")
    for b in books:
        print("-", b)

    print("\nВсего книг:", len(books))

In [42]:
# ==========================================
# 34. ПОСТРОЕНИЕ БАЗЫ ИЗ PDF
# ==========================================
def build_knowledge_base_from_pdf(
    pdf_path: str,
    source_name: str = None,
    max_pages_to_scan: int = 80,
    max_chars: int = 1200,
    overlap: int = 200,
    save: bool = True
):
    global GLOBAL_INDEX, GLOBAL_CHUNKS, debate_app

    if source_name is None:
        from pathlib import Path
        source_name = Path(pdf_path).stem

    log("=" * 80)
    log("СТАРТ ДОБАВЛЕНИЯ PDF В БАЗУ")
    log("PDF:", pdf_path)
    log("SOURCE:", source_name)
    log("=" * 80)

    start_time = time.time()

    doc = fitz.open(pdf_path)
    log("Всего страниц в PDF:", len(doc))

    chunks = build_chunks_from_pdf(
        doc,
        source_name=source_name,
        max_pages_to_scan=max_pages_to_scan,
        max_chars=max_chars,
        overlap=overlap
    )

    if not chunks:
        log("Чанки не найдены.")
        return GLOBAL_INDEX, GLOBAL_CHUNKS

    if chunks:
        log("Пример первого чанка:")
        log("pages:", chunks[0].get("pages"))
        log("metadata:", chunks[0].get("metadata"))
        log("text preview:", chunks[0].get("text", "")[:300])

    enriched_chunks, vectors = build_embeddings_for_chunks(chunks)

    # если базы ещё нет — создаём новую
    if GLOBAL_INDEX is None or len(GLOBAL_CHUNKS) == 0:
        GLOBAL_INDEX = build_faiss_index(vectors)
        GLOBAL_CHUNKS = enriched_chunks.copy()
    else:
        # иначе добавляем в существующую
        GLOBAL_INDEX.add(vectors)
        GLOBAL_CHUNKS.extend(enriched_chunks)

    if save:
        save_knowledge_base(GLOBAL_INDEX, GLOBAL_CHUNKS)

    debate_app = build_debate_graph()

    elapsed = time.time() - start_time

    log("=" * 80)
    log("БАЗА ГОТОВА")
    log("Всего чанков:", len(GLOBAL_CHUNKS))
    log("Векторов в индексе:", GLOBAL_INDEX.ntotal)
    log("Время:", round(elapsed / 60, 2), "мин")
    log("=" * 80)

    return GLOBAL_INDEX, GLOBAL_CHUNKS

In [43]:
# ==========================================
# 35. ЗАГРУЗИТЬ ГОТОВУЮ БАЗУ
# ==========================================
def load_ready_knowledge_base(index_path=INDEX_PATH, chunks_path=CHUNKS_PATH):
    global GLOBAL_INDEX, GLOBAL_CHUNKS, debate_app

    GLOBAL_INDEX, GLOBAL_CHUNKS = load_knowledge_base(index_path, chunks_path)
    debate_app = build_debate_graph()

    return GLOBAL_INDEX, GLOBAL_CHUNKS

In [44]:
# # ==========================================
# # 36. ПРИМЕР ИСПОЛЬЗОВАНИЯ
# # ==========================================
# pdf_path = r'D:\КНИГИ\book_torr\0568 C. Suryanarayana, A Inoue - Bulk Metallic Glasses - 2011.pdf'

# index, chunks = build_knowledge_base_from_pdf(
#     pdf_path=pdf_path,
#     source_name=os.path.splitext(os.path.basename(pdf_path))[0],
#     max_pages_to_scan=80,
#     max_chars=1200,
#     overlap=200,
#     save=True
# )

In [45]:
list_books()


Книги в базе:

- 0370 Steven G Johnson - Photonic Crystals From Theory to Practice - 2001
- 0568 C. Suryanarayana, A Inoue - Bulk Metallic Glasses - 2011
- 0618 Cai W., Shalaev V. Optical Metamaterials Fundamentals and Applications - 2010
- 0831 R. Marques, F. Martin, M. Sorolla Metamaterials with Negative Parameters Theory, Design, and Microwave Applications - 2008
- 0853 Stefano Salvatore Optical Metamaterials by Block Copolymer Self-Assembly - 2015
- 0936 Kedar Khare Fourier Optics and Computational Imaging 2016
- 0956 Neil Collings Fourier Optics in Image Processing 2019

Всего книг: 7


In [46]:
# results = search_in_db("Предложи технологию, чтобы сделать невидимым человеческому глазу предмет размером несколько метров с помощью метаматериалов", k=5)
# for r in results:
#     print("=" * 80)
#     print("SOURCE:", r["source"])
#     print("CHAPTER:", r["chapter"])
#     print("SECTION:", r["section"])
#     print("SUBSECTION:", r["subsection"])
#     print("PAGES:", r["pages"])
#     print("FORMULAS:", r["formulas"][:3])
#     print(r["text"][:800])

In [47]:
# start_ans = time.time()
# answer = ask_debate_rag("Предложи технологию, чтобы сделать невидимым человеческому глазу предмет размером несколько метров с помощью метаматериалов")
# print(answer)
# print('Длительность получение ответа: ', round((time.time() - start_ans) / 60, 1), 'мин.')

**ГИПОТЕЗА**  
Сделать объект размером несколько метров невидимым можно, покрыв его слоем метаматериала, у которого эффективный показатель преломления в видимом диапазоне (400‑700 нм) точно совпадает с показателем воздуха ( $n_{\text{eff}}\approx1$). Метаматериал должен быть построен из субволновой микроструктуры (например, гироид‑структуры, получаемой самосборкой блок‑кополимеров, с характерными размерами элементов $<10$ нм). При правильном подборе геометрии и параметров микроструктуры (возможно, с использованием фотонных кристаллов низкой‑мерной периодичности) достигается «клиновидное» отклонение световых лучей вокруг покрытого объекта, при этом свет не отражается и не поглощается, а выходит из‑за границы объекта в том же направлении, что и входил. Таким образом, для наблюдателя объект становится оптически невидимым.  

---

**КРИТИКА ГИПОТЕЗЫ**  

| Аспект гипотезы | Что подтверждено в контексте | Что слабо подтверждено / не подтверждено | Противоречия | Недостающие данные |
|----------------|------------------------------|------------------------------------------|--------------|--------------------|
| **Субволновая микроструктура (гиоид < 10 нм)** | Фрагмент 3 описывает гироид‑структуру с элементами $<10$ нм, обеспечивающую субволновое взаимодействие. | Нет данных о том, что такая структура даёт $n_{\text{eff}}\approx1$ без потерь. | Фрагмент 3 подчёркивает, что оптические метаматериалы «радикально труднее» произвести. | Экспериментальные измерения $n_{\text{eff}}(\lambda)$ и коэффициента потерь $\alpha(\lambda)$. |
| **Совпадение $n_{\text{eff}}$ с воздухом → отсутствие отражения/поглощения** | Теоретически возможно (эффективная среда). | В тексте нет подтверждения достижимости $n_{\text{eff}}=1$ и $\alpha\approx0$ в видимом диапазоне. | Фрагмент 3 указывает на высокие потери в оптических метаматериалах. | Данные о спектральных потерях, поляризационной независимости. |
| **Покрытие объекта размером несколько метров** | Нет упоминаний о масштабировании. | Никаких доказательств, что технологии (самосборка блок‑кополимеров, фотонные кристаллы) могут быть применены к площадям десятков‑сотен м². | Фрагмент 3 ограничивает текущие образцы микроскопическими размерами. | Технология массового производства однородных субволновых слоёв на метрах. |
| **Использование фотонных кристаллов низкой‑мерной периодичности** | Фрагмент 2 содержит обширный список работ по фотонным кристалам и их режимам. | Нет конкретных примеров «клиновидного» отклонения света вокруг крупного объекта. | Фрагменты 2 и 3 описывают микроскопические образцы, а не макроскопический клоакинг. | Моделирование (FEM, FDTD) эффективности «клоакинга» для метра‑размера; экспериментальные прототипы. |
| **Общая идея «оптической невидимости»** | Фрагмент 3 упоминает возможность создания «cloaking device». | Отсутствуют количественные подтверждения (угловой диапазон, полоса пропускания, фазовая согласованность). | Фрагмент 3 подчёркивает сложность реализации в видимом диапазоне. | Полный набор оптических характеристик: спектр, угол, поляризация, временная стабильность. |

**ИТОГ** – гипотеза согласуется с теоретической базой метаматериалов и фотонных кристаллов, но в предоставленных фрагментах нет экспериментальных или численных доказательств её реализуемости для объектов метрового масштаба, а также отсутствуют данные о достижении $n_{\text{eff}}=1$ без потерь.  

---

**ДОКАЗАТЕЛЬСТВА**  

*Подтверждает:*  
- **Фрагмент 3** – показывает, что метаматериалы могут «изменять оптические свойства вещества» и в принципе способны привести к «cloaking device», используя субволновую структуру (гиоид < 10 нм).  
- **Фрагмент 2** – перечисляет многочисленные работы по фотонным кристалам, их «guided modes», «confinement» и другим эффектам, что подтверждает наличие теоретической базы для управления распространением света.  

*Ослабляет или опровергает:*  
- **Фрагмент 3** – подчёркивает, что «метаматериал, активный в оптическом диапазоне (400‑700 нм), радикально труднее произвести» из‑за требуемого масштаба микроструктуры; нет данных о $n_{\text{eff}}\approx1$ и низких потерях.  
- **Фрагмент 3** – не содержит информации о масштабировании наноструктур до покрытий площадью «несколько метров».  
- **Фрагмент 2** – хотя и описывает фотонные кристалы, в представленных отрывках нет упоминания о практической реализации «клиновидного» отклонения света вокруг объектов метрового размера.  
- **Фрагменты 1, 4, 5** – не дают доказательств, связанных с метаматериалами для невидимости крупномасштабных объектов.  

---

**ФОРМУЛЫ ИЗ ИСТОЧНИКОВ**  

1. **Эффективный показатель преломления метаматериала** (эффективная среда)  
   $$
   n_{\text{eff}} = \sqrt{\varepsilon_{\text{eff}} \, \mu_{\text{eff}}}
   $$  
   где $\varepsilon_{\text{eff}}$ и $\mu_{\text{eff}}$ – эффективные диэлектрическая и магнитная проницаемости, получаемые из субволновой микроструктуры.  

2. **Трансформационная оптика (координатное преобразование)** – основной принцип «клоакинга»:  
   $$
   \varepsilon' = \frac{\Lambda \, \varepsilon \, \Lambda^{\mathrm{T}}}{\det \Lambda}, \qquad
   \mu' = \frac{\Lambda \, \mu \, \Lambda^{\mathrm{T}}}{\det \Lambda},
   $$  
   где $\Lambda$ – якобиан преобразования координат, задающего «выворачивание» световых траекторий вокруг скрываемого объёма.  

3. **Условие отсутствия отражения на границе** (импедансное согласование):  
   $$
   Z_{\text{eff}} = \sqrt{\frac{\mu_{\text{eff}}}{\varepsilon_{\text{eff}}}} \approx Z_{0},
   $$  
   где $Z_{0}$ – импеданс свободного пространства. При $Z_{\text{eff}}=Z_{0}$ отражение минимально.  

---

**ИСТОЧНИКИ**  

- **[Фрагмент 1]** Источник: 0956 Neil Collings *Fourier Optics in Image Processing* 2019 | Глава: Acronyms | Страницы: 49.  
- **[Фрагмент 2]** Источник: 0370 Steven G. Johnson *Photonic Crystals From Theory to Practice* 2001 | Глава: Concluding Remarks | Страницы: 159.  
- **[Фрагмент 3]** Источник: 0853 Stefano Salvatore *Optical Metamaterials by Block Copolymer Self-Assembly* 2015 | Глава: — | Страницы: 11.  
- **[Фрагмент 4]** Источник: 0956 Neil Collings *Fourier Optics in Image Processing* 2019 | Глава: Size | Страницы: 146.  
- **[Фрагмент 5]** Источник: 0568 C. Suryanarayana, A. Inoue *Bulk Metallic Glasses* 2011 | Глава: Applications | Страницы: 535.  

---

**ИТОГОВЫЙ ВЫВОД**  
Теоретически покрытие крупного объекта субволновым метаматериалом с $n_{\text{eff}}\approx1$ и импедансным согласованием может обеспечить «клиновидное» отклонение света и тем самым сделать объект невидимым. В представленных фрагментах подтверждены: (1) возможность создания субволновой гироид‑структуры (< 10 нм) и (2) наличие развитой теории фотонных кристаллов, позволяющей формировать требуемые пространственные распределения $\varepsilon$ и $\mu$. Однако отсутствуют экспериментальные данные, подтверждающие (а) достижение $n_{\text{eff}}=1$ без потерь в видимом диапазоне, (b) масштабирование технологии до покрытий площадью метров, и (c) практическую реализацию трансформационной оптики для таких размеров. Следовательно, гипотеза остаётся **спекулятивной** и требует дальнейших исследований в области:  

1. Нанофабрикации однородных субволновых слоёв на метрах.  
2. Измерения эффективных оптических параметров ( $n_{\text{eff}}$, $\alpha$ ) в видимом спектре.  
3. Численного моделирования (FDTD/FEM) полного 3‑D клоакинга для метра‑размера.  

---

**УРОВЕНЬ УВЕРЕННОСТИ**  
низкий – гипотеза согласуется с теоретической базой, но в текущих источниках нет достаточных экспериментальных или количественных подтверждений её практической реализуемости для объектов нескольких метров.

In [48]:
# ==========================================
# 37. ФУНКЦИЯ ЗАПРОСА К CHATGPT
# ==========================================
def chatgpt_chat(system: str, user: str, model: str = "gpt-5.4") -> str:

    response = client.responses.create(
        model=model,
        instructions=system,
        input=user
    )

    return response.output_text.strip()

In [49]:
# ==========================================
# 38. УЗЕЛ: ПОЛУЧЕНИЕ КОНТЕКСТА ИЗ CHATGPT
# ==========================================
def retrieve_context_from_chatgpt_node(state: dict) -> dict:
    
    question = state["question"]
    log("Debate node: retrieve_context_from_chatgpt")

    system = """
Ты аналитик-исследователь.

Твоя задача:
- прочитать вопрос пользователя;
- дать предварительный контекст по теме;
- выделить ключевые факты, идеи, ограничения и возможные подходы;
- не писать лишнюю воду;
- если уверенность низкая, прямо скажи об этом;
- пиши строго на русском языке.

Формат ответа:
КОНТЕКСТ
...
"""

    user = f"""
Вопрос:
{question}

Собери предварительный аналитический контекст по этому вопросу.
"""

    context = chatgpt_chat(system=system, user=user)

    return {
        "retrieved_items": [
            {
                "source": "ChatGPT",
                "text": context
            }
        ],
        "retrieved_context": context
    }

In [50]:
# ==========================================
# 39. ФУНКЦИЯ ЗАПУСКА ДЕБАТОВ ЧЕРЕЗ CHATGPT
# ==========================================
def ask_debate_chatgpt(question: str) -> str:

    log("Запуск debate ChatGPT mode...")

    state = {
        "question": question,
        "retrieved_context": "",
        "retrieved_items": [],
        "hypothesis": "",
        "criticism": "",
        "evidence": "",
        "final_answer": ""
    }

    state.update(retrieve_context_from_chatgpt_node(state))   
    state.update(build_hypothesis_node(state))
    state.update(critic_review_node(state))
    state.update(evidence_check_node(state))
    state.update(final_conclusion_node(state))

    return state["final_answer"]

In [51]:
# answer = ask_debate_chatgpt(
#     "Как погодные условия (дождь, снег, температура и влажность воздуха, облачность, туман и тп) могут повлиять на качество сотовой связи?"
# )

# print(answer)

**ГИПОТЕЗА**  
Погодные условия могут ухудшать качество сотовой связи как прямым воздействием на радиоволну (поглощение, рассеяние, изменение траектории распространения), так и косвенно через повреждение и сбои инфраструктуры (обледенение и снегонагрузка антенн, перебои питания, повреждения радиорелейных линий, усиление электромагнитных помех). Сила влияния сильно зависит от используемого частотного диапазона: на низких (≈ 600–900 МГц) и средних (≈ 1.8–2.6 ГГц) частотах дождь, снег и влажность вызывают лишь умеренное или слабое ослабление, тогда как на верхних средних (≈ 3.5 ГГц) и миллиметровых волнах (≥ 24 ГГц) осадки, туман, высокая влажность и обледенение приводят к заметному затуханию сигнала, падению SINR/RSRQ, снижению скорости и росту числа разрывов.

---

**КРИТИКА ГИПОТЕЗЫ**  
- В контексте указано, что для обычной LTE‑связи влияние погоды часто «не катастрофическое», что снижает практическую значимость гипотезы в массовых диапазонах.  
- Туман и обычная облачность описаны как «обычно слабое влияние», что противоречит части гипотезы, где они рассматриваются как потенциальные факторы ухудшения.  
- Отсутствие количественных данных (например, dB‑потерь при конкретных интенсивностях осадков) делает оценку «умеренное/сильное» субъективной.  

---

**ДОКАЗАТЕЛЬСТВА**  

*Подтверждает:*  
- «Погода может влиять на качество сотовой связи, но влияние сильно зависит от диапазона частот…» – подтверждает наличие как прямого, так и косвенного воздействия.  
- «Дождь — один из самых важных факторов, особенно на высоких частотах» – указывает на механизм поглощения и рассеяния.  
- «Снег… особенно мокрый снег и обледенение» – демонстрирует косвенный путь (накопление на антеннах).  
- «Туман обычно небольшое влияние, но на высоких частотах может быть заметнее» – подтверждает частотную зависимость.  
- «Влажность воздуха… может усиливать неблагоприятные эффекты на высоких частотах, способствовать конденсации в элементах инфраструктуры» – показывает двойное влияние.  
- «Температура влияет косвенно: перегрев оборудования, проблемы с аккумуляторами, обледенение, инверсии, меняющие траекторию распространения» – описывает косвенные механизмы.  
- «Ветер может раскачивать антенные опоры, разъюстировать радиорелейные антенны, приводить к механическим повреждениям и отключениям питания» – подтверждает инфраструктурный путь.  
- «Гроза… приводит к импульсным электромагнитным помехам, сбоям питания, выходу из строя оборудования» – усиливает косвенное влияние.  
- «Если пользователь находится на краю соты, в помещении с плохим проникновением, даже небольшое погодное ослабление становится заметным» – показывает, что эффект усиливается при небольшом запасе сигнала.  

*Ослабляет или опровергает:*  
- «Для обычной LTE‑связи влияние часто не катастрофическое, а лишь локальный и умеренный эффект» – уменьшает практическую значимость гипотезы в типичных диапазонах.  
- «Облачность обычно слабо влияет на массовую сотовую связь» – снижает роль облачности как фактора.  
- «Туман обычно небольшое влияние, особенно для обычных LTE‑диапазонов» – ослабляет утверждение о значимости тумана.  
- «Снег… влияние обычно слабее, чем дождя, если снег сухой и рыхлый» – уменьшает вес снежных осадков.  
- «Влажность воздуха реже вызывает резкое ухудшение в низких/средних диапазонах» – снижает роль влажности в этих диапазонах.  

---

**ФОРМУЛЫ ИЗ ИСТОЧНИКОВ**  

- Коэффициент осадочного затухания (rain attenuation)  

$$
\gamma = k \, R^{\alpha} \quad [\text{dB/km}]
$$  

где $R$ — интенсивность осадков (мм/ч), $k$ и $\alpha$ — параметры, зависящие от частоты.  

- Полный link‑budget с учётом осадочного затухания  

$$
P_{\text{rx}} = P_{\text{tx}} + G_{\text{tx}} + G_{\text{rx}} - L_{\text{fs}} - L_{\text{rain}}
$$  

- SINR (отношение полезного сигнала к сумме помех и шума)  

$$
\text{SINR} = \frac{P_{\text{rx}}}{I + N}
$$  

- Пороговое значение SINR, при котором меняется схема модуляции (пример для LTE)  

$$
\text{SINR}_{\text{threshold}}^{\text{QPSK}} \approx -3\ \text{dB}, \qquad
\text{SINR}_{\text{threshold}}^{\text{64QAM}} \approx 15\ \text{dB}
$$  

---

**ИСТОЧНИКИ**  
- [Фрагмент 1] Источник: *Контекст* | Глава: *Краткий вывод* | Раздел: *Влияние частот* | Страницы: *—*  
- [Фрагмент 2] Источник: *Контекст* | Глава: *Дождь* | Раздел: *Механизмы* | Страницы: *—*  
- [Фрагмент 3] Источник: *Контекст* | Глава: *Снег* | Раздел: *Обледенение* | Страницы: *—*  
- [Фрагмент 4] Источник: *Контекст* | Глава: *Туман* | Раздел: *Влияние на высокие частоты* | Страницы: *—*  
- [Фрагмент 5] Источник: *Контекст* | Глава: *Влажность* | Раздел: *Косвенные эффекты* | Страницы: *—*  
- [Фрагмент 6] Источник: *Контекст* | Глава: *Температура* | Раздел: *Косвенные влияния* | Страницы: *—*  
- [Фрагмент 7] Источник: *Контекст* | Глава: *Ветер* | Раздел: *Механические нагрузки* | Страницы: *—*  
- [Фрагмент 8] Источник: *Контекст* | Глава: *Гроза* | Раздел: *ЭМП‑помехи* | Страницы: *—*  

---

**ИТОГОВЫЙ ВЫВОД**  
Погодные явления способны ухудшать работу сотовых сетей двумя путями. Прямое ослабление радиоволн (особенно дождём и мокрым снегом) становится заметным при частотах выше ≈ 3 ГГц, где коэффициент затухания $\gamma$ растёт почти пропорционально $R^{\alpha}$. На низких и средних диапазонах влияние дождя и снега обычно ограничивается несколькими дБ, что часто компенсируется запасом по покрытию. Косвенные эффекты – обледенение антенн, перегрев оборудования, перебои питания, повреждения радиорелейных линий, ветровые колебания и грозовые помехи – могут приводить к падению SINR, переключению на более надёжные схемы модуляции и, в крайнем случае, к разрывам соединения независимо от частоты. Поэтому в практических оценках качества связи необходимо учитывать как спектральный диапазон, так и состояние инфраструктуры (запас сигнала, плотность базовых станций, наличие резервных систем). При высоких частотах и в условиях сильных осадков, низких температур с обледенением или сильных ветров/гроз ожидается существенное ухудшение, тогда как в умеренных климатических условиях и при работе в низких диапазонах влияние погоды обычно остаётся незначительным.

---

**УРОВЕНЬ УВЕРЕННОСТИ**  
высокий (текстовый контекст содержит согласованные описания механизмов, а формулы соответствуют общепринятым моделям распространения; однако количественная оценка требует дополнительных измерений).

In [54]:
import subprocess

djvu_path = r'D:\КНИГИ\book_torr\0108 M Born, E Wolf - Principles of Optics 6th edition - 1986.djvu'
djvutxt_path = r"D:\Программы\DjVuLibre-3.5.17-win32\djvutxt.exe"

result = subprocess.run(
    [djvutxt_path, djvu_path],
    capture_output=True,
    text=True,
    encoding="utf-8"
)

print(result.stdout)

CHAPTER 1 
BASIC PROPERTIES OF THE 
ELECTROMAGNETIC FIELD 
1.1 THE ELECTROMAGNETIC FIELD 
1.1.1 Maxwell's equations 
TE stae of excitation which is established in space by the presence of electric 
charges is said to cotitute an ectronetfie. It is represented by two vectors, 
g and B, ca the eric v and the netic iuct respectively.* 
To d e effect of the field on marial objects, it is neceary to introduce 
a second set of vec, v. e dtri cue deity j,  etric die D, 
and t   H. 
The space and time derivatives oftha five vectors are re.ted by Madl's , 
wch hold at every pot  whose neighborhood the physical propeies of the 
medium a continuous: 
crl  --    
= --j, (]) 
cl + = O, (2) 
the dot denotg differentiation with respect to time. 
Tey are supplement by two scar relation: 
• v D = p, (3) 
• v B = 0. (4) 
Eq. (3) may  reaed  a defiling eqatio fbr ghe electric char densi$ 7 p and (4) 
y  said  imply that no free magnetic pol

In [60]:
def extract_text_from_djvu(djvu_path: str, djvutxt_path: str):    

    result = subprocess.run(
        [djvutxt_path, djvu_path],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )   

    text = result.stdout.strip()
    return print(text)

In [62]:
extract_text_from_djvu(djvu_path, djvutxt_path)

CHAPTER 1 
BASIC PROPERTIES OF THE 
ELECTROMAGNETIC FIELD 
1.1 THE ELECTROMAGNETIC FIELD 
1.1.1 Maxwell's equations 
TE stae of excitation which is established in space by the presence of electric 
charges is said to cotitute an ectronetfie. It is represented by two vectors, 
g and B, ca the eric v and the netic iuct respectively.* 
To d e effect of the field on marial objects, it is neceary to introduce 
a second set of vec, v. e dtri cue deity j,  etric die D, 
and t   H. 
The space and time derivatives oftha five vectors are re.ted by Madl's , 
wch hold at every pot  whose neighborhood the physical propeies of the 
medium a continuous: 
crl  --    
= --j, (]) 
cl + = O, (2) 
the dot denotg differentiation with respect to time. 
Tey are supplement by two scar relation: 
• v D = p, (3) 
• v B = 0. (4) 
Eq. (3) may  reaed  a defiling eqatio fbr ghe electric char densi$ 7 p and (4) 
y  said  imply that no free magnetic pol